In [1]:
from pyspark.sql import SparkSession

In [2]:
import pyspark.sql.functions as F
import pyspark.sql.types as T

from datetime import datetime
from datetime import timedelta
from dateutil.relativedelta import relativedelta

import logging

from IPython.core.display import display, HTML
display(HTML("<style>pre { white-space: pre !important; }</style>"))

/tmp/ipykernel_60/3320559344.py:10: DeprecationWarning: Importing display from IPython.core.display is deprecated since IPython 7.14, please import from IPython display
  from IPython.core.display import display, HTML


In [3]:
spark = SparkSession.builder \
    .appName("mob2") \
    .master("spark://spark-master:7077") \
    .config("spark.sql.catalogImplementation", "hive") \
    .config("spark.sql.warehouse.dir", "hdfs://namenode:9000/user/hive/warehouse") \
    .config("spark.cores.max", "8")\
    .enableHiveSupport() \
    .getOrCreate()

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/09 17:30:28 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [9]:
def build_customer_sex_age_income(spark, logical_date, logger):
    from pyspark.sql import functions as F

    business_month_end = logical_date + relativedelta(months=1, days=-1)

    logger.info('Processing customer_sex ...')
    model_sex = (
        spark.table('ff_dm.scoring_customer_sex')
        .select('msisdn', F.col('model_sex'))
    )
    logger.info('Processing customer_age ...')
    model_age = (
        spark.table('ff_dm.scoring_customer_age')
        .select('msisdn', F.col('model_age'))
    )
    logger.info('Processing customer_income ...')
    # можно заменить для будущих расчетов на gr_dm.dm_model_income_v5__hist
    income = (
        spark.table('gr_dm.dm_model_income__hist')
        .select('msisdn', F.col('model_income'))
    )
    logger.info('Joining tables ...')
    final_df = (
        model_sex
        .join(model_age, ['msisdn'], 'full')
        .join(income, ['msisdn'], 'full')
        .drop_duplicates(['msisdn'])
    )

    return final_df


In [10]:
log = logging.getLogger(__name__)

In [11]:
logical_date = datetime.now().date()

In [12]:
build_customer_sex_age_income(spark, logical_date, log)

DataFrame[msisdn: int, model_sex: int, model_age: int, model_income: int]

In [13]:
build_customer_sex_age_income(spark, logical_date, log).count()

2505963

In [15]:
spark.stop()